In [40]:
import os
import torch
import pytorch_lightning as pl
from transformers import get_scheduler, AutoModelForCausalLM, AutoProcessor, AutoConfig
from florence2_large import processing_florence2
from peft import LoraConfig, get_peft_model, PeftModel
from pytorch_lightning import Trainer
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd
import numpy as np
import ast
import torchvision.transforms as T
from PIL import Image

# os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [41]:
class PadChestDataset(Dataset):
    def __init__(self, csv_path, root, transform=None, label_binarizer=None):
        try:
            self.data_info = pd.read_csv(csv_path)
        except Exception as e:
            raise ValueError(f"Error reading CSV file at {csv_path}: {e}")

        self.root = root
        self.transform = transform
        self.label_binarizer = label_binarizer

    def __getitem__(self, index):
        print("getting image")
        # Load and preprocess image
        img_path = os.path.join(self.root, self.data_info.iloc[index]['ImageID'])
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception as e:
            raise ValueError(f"Error loading image at {img_path}: {e}")

        # if self.transform:
        #     img = self.transform(img)

        # Process labels
        labels_str = self.data_info.iloc[index]['Labels']
        labels = ast.literal_eval(labels_str)
        labels = [label.strip().lower() for label in labels]
        binary_labels = self.label_binarizer.transform([labels])[0]

        return img, binary_labels

    def __len__(self):
        return len(self.data_info)

In [42]:
# defines a transform to convert PIL images to tensors
transform = T.Compose([
    T.Resize((224, 224)),  # resize images to 224x224
    T.ToTensor()
])

# def collate_fn(batch):
#     images, labels = zip(*batch)
#     images = torch.stack(images)  # Combine images into a single tensor
#     labels = torch.tensor(labels, dtype=torch.float32)  # Combine labels
#     return images, labels

def collate_fn(batch):
    # Unpack batch into separate lists
    images, labels = zip(*batch)
    
    # resize and convert images to tensors
    images = torch.stack([transform(img) for img in images])

    # convert labels to a single numpy array, then to a pytorch tensor
    # otherwise, everything runs really slowly
    labels = torch.tensor(np.array(labels), dtype=torch.float32)
    
    return images, labels

In [43]:
class PadChestDataLoaderManager(pl.LightningDataModule):
    def __init__(self, config):
        super().__init__()
        self.img_root = config.get("img_root")
        self.annotation_csv = config.get("annotation_csv")
        self.batch_size = config.get("batch_size", 8)
        self.num_workers = config.get("num_workers", 0)

        # Prepare the label binarizer
        self.label_binarizer = MultiLabelBinarizer()
        df = pd.read_csv(self.annotation_csv)
        all_labels = [ast.literal_eval(labels) for labels in df['Labels']]
        self.label_binarizer.fit(all_labels)

    def create_dataloader(self, split):
        base_dataset = PadChestDataset(
            csv_path=self.annotation_csv,
            root=self.img_root,
            transform=None,
            label_binarizer=self.label_binarizer
        )

        # for i in range(5):
        #     img, labels = base_dataset[i]
        #     print(f"Image shape: {img.size}, Labels: {labels}")

        return DataLoader(
            base_dataset,
            batch_size=self.batch_size,
            collate_fn=collate_fn,
            num_workers=self.num_workers,
            shuffle=(split == 'train')
        )

    def train_dataloader(self):
        return self.create_dataloader(split='train')

    def val_dataloader(self):
        return self.create_dataloader(split='val')

    def test_dataloader(self):
        return self.create_dataloader(split='test')

In [44]:
class FlorenceLightningModel(pl.LightningModule):
    def __init__(self, model, processor, lr=1e-6, num_training_steps=None):
        super(FlorenceLightningModel, self).__init__()
        self.model = model
        self.processor = processor
        self.lr = float(lr)
        self.num_training_steps = num_training_steps

    def training_step(self, batch, batch_idx):
        # images, labels = batch
        # outputs = self.model(images)
        # loss_fn = torch.nn.BCEWithLogitsLoss()
        # loss = loss_fn(outputs, labels)
        # self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, batch_size=len(images))
        # return loss
        images, labels = batch
        inputs = self.processor(
            text=questions, 
            images=images, 
            return_tensors="pt", 
            padding=True
        ).to(self.device)

        inputs["input_ids"] = inputs["input_ids"].long()

        print(f"Input IDs dtype: {inputs['input_ids'].dtype}")
        print(f"Pixel Values dtype: {inputs['pixel_values'].dtype}")
        print(f"Labels dtype: {labels.dtype}")
        print(inputs["input_ids"].dtype)  # Should be torch.long

        outputs = self.model(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], labels=labels)
        loss = outputs.loss
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, batch_size=len(images))
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        outputs = self.model(images)
        loss_fn = torch.nn.BCEWithLogitsLoss()
        loss = loss_fn(outputs, labels)
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.lr)
        lr_scheduler = get_scheduler(
            name="linear",
            optimizer=optimizer,
            num_warmup_steps=0,
            num_training_steps=self.num_training_steps,
        )
        return [optimizer], [lr_scheduler]

In [45]:
config = {
    "img_root": "../padchest_images",
    "annotation_csv": "../padchest_labels_truncated.csv",
    "batch_size": 8,
    "num_workers": 0,
    "trainer": {"learning_rate": 1e-5, "max_epochs": 10},
}

In [46]:
data_loader_manager = PadChestDataLoaderManager(config)
train_dataloader = data_loader_manager.train_dataloader()
val_dataloader = data_loader_manager.val_dataloader()

dataset_size = len(train_dataloader.dataset)
num_training_steps = (dataset_size + config['batch_size'] - 1) // config['batch_size']

In [47]:
dataset = PadChestDataset(
    csv_path="../padchest_labels_truncated.csv",
    root="../padchest_images",
    label_binarizer=data_loader_manager.label_binarizer
)

### testing if collate_fn works properly
# for i in range(5):  # Check the first 5 items
#     try:
#         img, labels = dataset[i]
#         print(f"Image size: {img.size}, Labels: {labels}")
#     except Exception as e:
#         print(f"Error at index {i}: {e}")

# batch = [dataset[i] for i in range(4)]
# try:
#     images, labels = collate_fn(batch)
#     print(f"Batch image shape: {images.shape}, Batch label shape: {labels.shape}")
# except Exception as e:
#     print(f"Collate function error: {e}")



In [48]:
# ### checking if data is batched correctly

# for i, batch in enumerate(train_dataloader):
#     print(f"Batch {i}: Images shape {batch[0].shape}, Labels shape {batch[1].shape}")
#     if i == 5:
#         break


# for batch in train_dataloader:
#     print("batch")
#     images, labels = batch
#     print(f"Batch image shape: {images.shape}, Batch label shape: {labels.shape}")
#     break

In [49]:
MODEL_NAME = "microsoft/Florence-2-large"
config_model = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, config=config_model, trust_remote_code=True)
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)

In [50]:
lightning_model = FlorenceLightningModel(model=model, processor=processor, lr=config['trainer']['learning_rate'], num_training_steps=num_training_steps)

In [51]:
trainer = Trainer(
    max_epochs=config['trainer']['max_epochs'], 
    accelerator="auto", 
    devices="auto",
    num_sanity_val_steps=0)

trainer.fit(lightning_model, train_dataloader, val_dataloader)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type                              | Params | Mode
-------------------------------------------------------------------
0 | model | Florence2ForConditionalGeneration | 828 M  | eval
-------------------------------------------------------------------
828 M     Trainable params
0         Non-trainable params
828 M     Total params
3,315.941 Total estimated model params size (MB)
0         Modules in train mode
880       Modules in eval mode
C:\Users\Mark\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'train_dataloader' does not 

Epoch 0:   0%|          | 0/376 [00:00<?, ?it/s] getting image
getting image
getting image
getting image
getting image
getting image
getting image
getting image


NameError: name 'questions' is not defined